# ___Themeda triandra - Power Analysis___
------------------

In [1]:
!python --version

Python 3.13.9


The system cannot find the path specified.


In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from numba import njit

## ___Data wrangling___
-----------------

In [22]:
# load in the data for soil and climatic properties and root trait data
root_traits = pd.read_csv(r"../data/chapter3/vin_themeda_root_traits.csv", usecols=("Accession", "SRL", "RTD", "Diameter")) # units => m/g, g/cm3, mm
root_traits = root_traits.rename({_: _.lower().strip() for _ in root_traits.columns}, axis=1)
root_traits.loc[:, "accession"] = root_traits.accession.replace(
    {_: _.replace(' ', '_').strip() for _ in root_traits.accession.unique()}
).replace({"Mt_Fox_N_Park_QLD": "Mt_Fox_National_Park_QLD", "Sydney": "Sydney_NSW"}) # replace these two to match with the climate info dataset

root_traits.head()

,accession,srl,rtd,diameter
0,Dalby_QLD,16.292527,0.256367,0.5521
1,Dalby_QLD,35.587354,0.145137,0.4965
2,Dalby_QLD,26.271140,0.332818,0.3816
3,Dalby_QLD,17.405689,0.176030,0.6447
4,Mt_Fox_National_Park_QLD,35.094561,0.265127,0.3699


In [15]:
root_traits.accession.unique()

array(['Dalby_QLD', 'Mt_Fox_National_Park_QLD', 'Panawonica_WA',
       'Rainbow_Valley_NT', 'Sydney_NSW', 'Virginia_Gardens_SA'],
      dtype=object)

In [4]:
climate = pd.read_csv(r"../data/chapter3/site_metereology.csv", skiprows=range(2), usecols=("site", "state", "annual_rainfall_mm", "mean_annual_temp_celsius"))
climate.loc[:, "site"] = (climate.site + '_' + climate.state).str.replace(' ', '_')

In [77]:
climate

,site,state,annual_rainfall_mm,mean_annual_temp_celsius
0,Rainbow_Valley_NT,NT,189.3,28.9
1,Panawonica_WA,WA,404.4,34.7
2,Forbes_NSW,NSW,520.4,24.5
3,Hobart_TAS,TAS,611.4,17.0
4,Dalby_QLD,QLD,621.4,27.0
5,Virginia_Gardens_SA,SA,468.7,22.6
6,Mt_Fox_National_Park_QLD,QLD,642.8,29.3
7,Sydney_NSW,NSW,1156.9,23.4


In [5]:
# soil properties data
soil = pd.read_csv(r"../data/chapter3/CSBP_soil_analysis_Themeda_and_Sorghum.csv")# , usecols=("Customer Sample ID", "Sample Name 2", ""))
soil = soil.rename({_: _.lower().strip().replace(' ', '_').replace('%', "prcnt").replace('(', '').replace(')', '') for _ in soil.columns}, axis=1) # clean up the column names
soil = soil.drop(["lab_number", "date_received", "customer_sample_id", "sample_name_1", "latitude", "longitude", "depth", "colour", "gravel_percent", "texture"], axis=1) # drop useless columns
soil = soil.query("sample_name_2.isin(('Cobbler Ck SA', 'Dalby Qld', 'Mt Fox Qld', 'Pannawonica WA', 'Rainbow Valley NT', 'Hornsby Heights NSW'))").reset_index(drop=True) # filter the six needed rows
soil.loc[:, "sample_name_2"] = soil.sample_name_2.replace({ # update the site names to match the other datsets
    "Cobbler Ck SA": "Virginia_Gardens_SA", "Dalby Qld": "Dalby_QLD", "Hornsby Heights NSW": "Sydney_NSW", "Mt Fox Qld": "Mt_Fox_National_Park_QLD",
    "Pannawonica WA": "Panawonica_WA", "Rainbow Valley NT": "Rainbow_Valley_NT"
})

In [ ]:
# pH(H2O) vs pH(CaCl2)
# https://agriculture.vic.gov.au/farm-management/soil/understanding-soil-tests-for-pastures
# Soil pH CaCl2 values are usually between 0.5 to 1.1 units lower than pH (water). The pH (water) value readily reflects current soil conditions, but is subject to seasonal variations.
# The CaCl2 test is useful for long term monitoring of pH and is less subject to seasonal variations

In [ ]:
# # https://soilqualityknowledgebase.org.au/resources/soil-phosphorus-testing-colwell-p-and-dgt-p/

# Soil contains phosphorus that can be conceptualised as four pools: solution phosphorus, sorbed phosphorus, mineral phosphorus and organic phosphorus.
# The amount of phosphorus measured by a soil test is dependent on the amount of phosphorus present in each pool and the ability of the soil test method to extract phosphorus from each pool. 
# For the test to be a useful predictor of crop yield responses to fertiliser phosphorus, the test must measure phosphorus that is available to crops for the soil-crop system in question.

# Colewell method for soil P
# In a Colwell phosphorus test, an extract is obtained by adding a soil sample to a sodium bicarbonate solution adjusted to pH 8.5, and agitating for 16 hours. This extract is then acidified before its phosphorus 
# concentration is measured colorimetrically. The Colwell method measures sorbed and solution phosphorus, but does not provide any information on the equilibrium between these two pools, which is determined by the
# phosphors buffering capacity of the soil. The phosphorus buffering index (PBI) is used in combination with Colwell-P to assess the levels of soil P supply to crops and pastures. As the phosphorus buffering index 
# increases, the level of Colwell-P required to provide a sufficient level of phosphorus for crops and pastures increases.

<a src="https://soilqualityknowledgebase.org.au/wp-content/uploads/2023/11/soil-phosphorus-pools-diagram-simple_PNG.png">Image source</a><br><br>

<img src="./soil-phosphorus-pools-diagram-simple_PNG.png" width="400px">

In [39]:
def _string_to_float(value: str) -> float:
    """
    if the value is <1 let'set that to 0!! => this is an arbitrary decision
    """
    try:
        return float(value.strip())
    except ValueError:
        return 0.00000

In [41]:
soil.loc[:, "nitrate_nitrogen"] = soil.nitrate_nitrogen.apply(_string_to_float)
soil

,sample_name_2,ammonium_nitrogen,nitrate_nitrogen,phosphorus_colwell,potassium_colwell,sulfur,organic_carbon,conductivity,ph_level_cacl2,ph_level_h2o,total_nitrogen,total_phosphorus,total_carbon,prcnt_clay,prcnt_course_sand,prcnt_fine_sand,prcnt_sand,prcnt_silt
0,Virginia_Gardens_SA,2,0.0,5,511,10.7,2.28,0.092,6.1,7.0,0.22,258.6,3.07,31.69,18.82,29.01,47.83,20.48
1,Dalby_QLD,6,0.0,18,398,4,2.88,0.069,6.0,6.9,0.27,402.1,3.96,23.11,35.26,26.89,62.15,14.74
2,Sydney_NSW,9,0.0,4,206,4.2,3.26,0.052,4.9,5.9,0.18,229.2,4.04,13.88,51.89,23.51,75.4,10.72
3,Mt_Fox_National_Park_QLD,6,0.0,6,314,3.6,1.71,0.035,5.7,6.7,0.19,157.5,2.66,21.6,33.49,22.22,55.71,22.69
4,Panawonica_WA,2,13.0,8,385,12.8,1.53,0.232,7.2,8.0,0.14,184.2,2.27,25.64,6,29.82,35.82,38.54
5,Rainbow_Valley_NT,1,0.0,6,125,1.1,0.25,0.061,7.5,8.8,0.02,89.7,0.51,5.8,60.38,31.88,92.26,1.95


In [48]:
# add ploidy info as another column - got the data from Vin
ploidy_lookup = pd.Series(name="ploidy_n", data={
    "Dalby_QLD": 6,
    "Mt_Fox_National_Park_QLD": 4,
    "Panawonica_WA": 2,
    "Rainbow_Valley_NT": 4,
    "Sydney_NSW": 2,
    "Virginia_Gardens_SA": 4
})

In [104]:
# meta info - climate and soil properties
data = pd.merge(left=climate, left_on="site", right=soil, right_on="sample_name_2").drop(["state", "sample_name_2"], axis=1)
data = pd.merge(left=data, left_on="site", right=ploidy_lookup, right_index=True).reset_index(drop=True)

In [105]:
data.dtypes # why do we have so many object types????

site                         object
annual_rainfall_mm          float64
mean_annual_temp_celsius    float64
ammonium_nitrogen            object
nitrate_nitrogen             object
phosphorus_colwell           object
potassium_colwell            object
sulfur                       object
organic_carbon               object
conductivity                 object
ph_level_cacl2              float64
ph_level_h2o                float64
total_nitrogen               object
total_phosphorus             object
total_carbon                 object
prcnt_clay                   object
prcnt_course_sand            object
prcnt_fine_sand              object
prcnt_sand                   object
prcnt_silt                   object
ploidy_n                      int64
dtype: object

In [106]:
data.ploidy_n = data.ploidy_n.astype(np.float64)
data.iloc[:, 1:] = data.iloc[:, 1:].astype(np.float64)
data = pd.merge(left=data, left_on="site", right=root_traits.loc[:, ["accession", "srl", "diameter"]], right_on="accession").drop("accession", axis=1) # instead of grouping and using site averages, let's fit the
# model to all the data points
data

,site,annual_rainfall_mm,mean_annual_temp_celsius,ammonium_nitrogen,nitrate_nitrogen,phosphorus_colwell,potassium_colwell,sulfur,organic_carbon,conductivity,...,total_phosphorus,total_carbon,prcnt_clay,prcnt_course_sand,prcnt_fine_sand,prcnt_sand,prcnt_silt,ploidy_n,srl,diameter
0,Rainbow_Valley_NT,189.3,28.9,1.0,0.0,6.0,125.0,1.1,0.25,0.061,...,89.7,0.51,5.8,60.38,31.88,92.26,1.95,4.0,49.215762,0.394200
1,Rainbow_Valley_NT,189.3,28.9,1.0,0.0,6.0,125.0,1.1,0.25,0.061,...,89.7,0.51,5.8,60.38,31.88,92.26,1.95,4.0,22.585138,0.593900
2,Rainbow_Valley_NT,189.3,28.9,1.0,0.0,6.0,125.0,1.1,0.25,0.061,...,89.7,0.51,5.8,60.38,31.88,92.26,1.95,4.0,27.897760,0.411200
3,Rainbow_Valley_NT,189.3,28.9,1.0,0.0,6.0,125.0,1.1,0.25,0.061,...,89.7,0.51,5.8,60.38,31.88,92.26,1.95,4.0,22.529411,0.472000
4,Rainbow_Valley_NT,189.3,28.9,1.0,0.0,6.0,125.0,1.1,0.25,0.061,...,89.7,0.51,5.8,60.38,31.88,92.26,1.95,4.0,21.362401,0.634400
5,Panawonica_WA,404.4,34.7,2.0,13.0,8.0,385.0,12.8,1.53,0.232,...,184.2,2.27,25.64,6.0,29.82,35.82,38.54,2.0,21.241229,0.317000
6,Panawonica_WA,404.4,34.7,2.0,13.0,8.0,385.0,12.8,1.53,0.232,...,184.2,2.27,25.64,6.0,29.82,35.82,38.54,2.0,22.604293,0.694700
7,Panawonica_WA,404.4,34.7,2.0,13.0,8.0,385.0,12.8,1.53,0.232,...,184.2,2.27,25.64,6.0,29.82,35.82,38.54,2.0,15.638653,0.715900
8,Panawonica_WA,404.4,34.7,2.0,13.0,8.0,385.0,12.8,1.53,0.232,...,184.2,2.27,25.64,6.0,29.82,35.82,38.54,2.0,12.558968,0.759600
9,Dalby_QLD,621.4,27.0,6.0,0.0,18.0,398.0,4.0,2.88,0.069,...,402.1,3.96,23.11,35.26,26.89,62.15,14.74,6.0,16.292527,0.552100


In [107]:
data.shape

(24, 23)

In [ ]:
# maybe we shouldn't treat ploidy level as a continuous variable???

In [75]:
# what were the units of the soil properties??
pd.read_csv(r"../data/chapter3/CSBP_soil_analysis_Themeda_and_Sorghum.csv").loc[0, :].dropna()

Gravel_percent            %
Ammonium Nitrogen     mg/kg
Nitrate Nitrogen      mg/kg
Phosphorus Colwell    mg/kg
Potassium Colwell     mg/kg
Sulfur                mg/kg
Organic Carbon            %
Conductivity           dS/m
Total Nitrogen            %
Total Phosphorus      mg/kg
Total Carbon              %
% Clay                    %
% Course Sand             %
% Fine Sand               %
% Sand                    %
% Silt                    %
Name: 0, dtype: object

In [108]:
model = LinearRegression(n_jobs=16)
scaler = StandardScaler()

In [118]:
scaled_x = scaler.fit_transform(data.iloc[:, 1:21]) # all the independent variables
y = data.iloc[:, 21:] # SRL & RD

In [119]:
model.fit(X=scaled_x, y=y)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,16
,positive,False


In [120]:
model.coef_ # the first array is for SRL and the second is for RD

array([[ 5.69981143e-01, -2.00809027e+00, -1.51339582e-01,
        -9.29502498e-01, -1.64780879e+00,  1.76847675e-03,
         3.11138438e-01,  4.52683769e-01, -5.45854679e-01,
        -6.50447978e-01, -5.16001954e-01, -2.47201011e-01,
        -2.76501560e-01,  1.75951219e-01, -5.24222784e-02,
         5.29546047e-01,  3.72335602e-01,  5.86907116e-01,
        -9.21724919e-01, -8.29613045e-01],
       [-9.63435167e-04, -6.01010953e-03, -7.40743673e-03,
         7.84460683e-03,  1.39024947e-03,  4.92881835e-03,
         1.23344200e-02,  5.49274049e-03,  1.30047941e-02,
         5.45266706e-03,  4.05984870e-03, -7.44840639e-04,
         7.57836529e-03,  3.06738795e-03,  2.86576806e-03,
        -4.17329772e-03,  1.62156549e-02, -1.16940481e-03,
        -2.30238852e-04, -6.82558256e-03]])

In [121]:
np.abs(model.coef_) > 0.1

array([[ True,  True,  True,  True,  True, False,  True,  True,  True,
         True,  True,  True,  True,  True, False,  True,  True,  True,
         True,  True],
       [False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False]])

In [100]:
model.intercept_

array([27.78765886,  0.52507368])